# Compare Algorithms: IMPALA vs DreamerV2

Compare training curves and sample efficiency between IMPALA (V-trace) and DreamerV2 (world model) on Memory Maze.

**Requirements**: Completed training runs for both algorithms  
**Time**: ~5 minutes (plotting only, no training)

> **Don't have training logs yet?** Run `02_train_and_plot.ipynb` for IMPALA. For DreamerV2:
> ```bash
> python train_dreamer.py --num_envs 8 --total_steps 500000 --xpid dreamer_9x9_demo
> ```

In [ ]:
%matplotlib inline

In [ ]:
import sys, os

import pathlib
PROJECT_ROOT = str(pathlib.Path.cwd().parent) if pathlib.Path('../train_impala.py').exists() else str(pathlib.Path.cwd())
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'notebooks'))

import numpy as np
import matplotlib.pyplot as plt

from utils import load_training_logs, logs_to_arrays, plot_learning_curve, smooth

## 1. Load Training Logs

Set the experiment IDs to match your training runs.

In [ ]:
# Set these to match your experiment IDs
IMPALA_XPID = "impala_9x9_demo"
DREAMER_XPID = "dreamer_9x9_demo"
LOGDIR = os.path.expanduser("~/logs/torchbeast")

# List available experiments
print("Available experiments:")
if os.path.exists(LOGDIR):
    for d in sorted(os.listdir(LOGDIR)):
        logs_path = os.path.join(LOGDIR, d, "logs.csv")
        if os.path.exists(logs_path):
            size = os.path.getsize(logs_path)
            print(f"  {d}  (logs.csv: {size/1024:.1f} KB)")
else:
    print(f"  Log directory not found: {LOGDIR}")

In [ ]:
try:
    impala_rows = load_training_logs(IMPALA_XPID)
    impala_steps, impala_returns = logs_to_arrays(impala_rows)
    print(f"IMPALA: {len(impala_rows)} entries, {impala_steps[-1]:,.0f} steps")
except FileNotFoundError:
    impala_rows, impala_steps, impala_returns = None, None, None
    print(f"IMPALA logs not found for xpid='{IMPALA_XPID}'")

try:
    dreamer_rows = load_training_logs(DREAMER_XPID)
    dreamer_steps, dreamer_returns = logs_to_arrays(dreamer_rows)
    print(f"DreamerV2: {len(dreamer_rows)} entries, {dreamer_steps[-1]:,.0f} steps")
except FileNotFoundError:
    dreamer_rows, dreamer_steps, dreamer_returns = None, None, None
    print(f"DreamerV2 logs not found for xpid='{DREAMER_XPID}'")

## 2. Learning Curves Comparison

Plot IMPALA and DreamerV2 episode returns on the same axes. The x-axis is environment steps (not wall-clock time), which is the fair comparison metric.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

if impala_steps is not None:
    plot_learning_curve(impala_steps, impala_returns,
                        label="IMPALA (V-trace)", color="tab:blue",
                        smooth_window=20, ax=ax)

if dreamer_steps is not None:
    plot_learning_curve(dreamer_steps, dreamer_returns,
                        label="DreamerV2 (World Model)", color="tab:orange",
                        smooth_window=20, ax=ax)

ax.set_title("Learning Curves: IMPALA vs DreamerV2 — Memory Maze 9x9")
ax.set_xlabel("Environment Steps")
ax.set_ylabel("Episode Return")
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig("algorithm_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Sample Efficiency

Compare how quickly each algorithm reaches different performance thresholds. DreamerV2 is expected to be more sample-efficient due to its world model — it learns from imagined experience.

In [ ]:
thresholds = [1.0, 2.0, 3.0, 5.0, 8.0, 10.0]

print(f"{'Threshold':>12s}  {'IMPALA Steps':>15s}  {'DreamerV2 Steps':>15s}  {'Ratio':>8s}")
print("-" * 60)

for thresh in thresholds:
    impala_at = "N/A"
    dreamer_at = "N/A"
    ratio = ""

    if impala_steps is not None:
        smoothed_i = smooth(impala_returns, 20)
        idx = np.where(smoothed_i >= thresh)[0]
        if len(idx) > 0:
            impala_at = f"{impala_steps[idx[0]]:,.0f}"

    if dreamer_steps is not None:
        smoothed_d = smooth(dreamer_returns, 20)
        idx = np.where(smoothed_d >= thresh)[0]
        if len(idx) > 0:
            dreamer_at = f"{dreamer_steps[idx[0]]:,.0f}"

    if impala_at != "N/A" and dreamer_at != "N/A":
        ratio_val = float(impala_at.replace(",", "")) / float(dreamer_at.replace(",", ""))
        ratio = f"{ratio_val:.1f}x"

    print(f"{thresh:12.1f}  {impala_at:>15s}  {dreamer_at:>15s}  {ratio:>8s}")

## 4. Paper Reference Scores

For context, here are the oracle and baseline scores from the Memory Maze paper (Table 3).

| Maze | Oracle | DreamerV2 (paper) | IMPALA (paper) |
|------|--------|-------------------|----------------|
| 9x9  | 26.4 | 15.4 | 11.6 |
| 11x11 | 44.3 | 16.7 | 9.5 |
| 13x13 | 55.5 | 14.6 | 5.5 |
| 15x15 | 67.7 | 12.0 | 3.1 |

These scores are at 100M environment steps with 128 parallel environments.

In [ ]:
maze_sizes = ["9x9", "11x11", "13x13", "15x15"]
oracle_scores = [26.4, 44.3, 55.5, 67.7]
paper_dreamer = [15.4, 16.7, 14.6, 12.0]
paper_impala = [11.6, 9.5, 5.5, 3.1]

x = np.arange(len(maze_sizes))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, oracle_scores, width, label="Oracle", color="tab:green", alpha=0.7)
ax.bar(x, paper_dreamer, width, label="DreamerV2 (paper)", color="tab:orange", alpha=0.7)
ax.bar(x + width, paper_impala, width, label="IMPALA (paper)", color="tab:blue", alpha=0.7)

ax.set_xlabel("Maze Size")
ax.set_ylabel("Mean Episode Return (100M steps)")
ax.set_title("Memory Maze: Paper Results by Maze Size")
ax.set_xticks(x)
ax.set_xticklabels(maze_sizes)
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("paper_scores.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. DreamerV2: TBTT vs Standard

If you trained DreamerV2 with and without `--tbtt` (truncated backprop through time), compare them here. TBTT carries the RSSM state across sequential batches, which should help with long-horizon credit assignment.

In [ ]:
DREAMER_TBTT_XPID = "dreamer_9x9_tbtt_demo"  # Set to your TBTT experiment

try:
    tbtt_rows = load_training_logs(DREAMER_TBTT_XPID)
    tbtt_steps, tbtt_returns = logs_to_arrays(tbtt_rows)
    has_tbtt = True
    print(f"TBTT logs loaded: {len(tbtt_rows)} entries")
except FileNotFoundError:
    has_tbtt = False
    print(f"No TBTT experiment found (xpid='{DREAMER_TBTT_XPID}')")
    print("Train one with: python train_dreamer.py --tbtt --xpid dreamer_9x9_tbtt_demo")

if has_tbtt and dreamer_steps is not None:
    fig, ax = plt.subplots(figsize=(10, 5))
    plot_learning_curve(dreamer_steps, dreamer_returns,
                        label="DreamerV2 (standard)", color="tab:orange",
                        smooth_window=20, ax=ax)
    plot_learning_curve(tbtt_steps, tbtt_returns,
                        label="DreamerV2 (TBTT)", color="tab:red",
                        smooth_window=20, ax=ax)
    ax.set_title("DreamerV2: Standard vs TBTT")
    ax.legend(fontsize=12)
    plt.tight_layout()
    plt.show()

## 6. Wall-Clock Efficiency

Environment steps are the fair comparison, but wall-clock time matters too. IMPALA uses many parallel actors (128 default) while DreamerV2 uses fewer environments (8 default) but trains a world model.

In [ ]:
if impala_rows and dreamer_rows:
    # Extract timestamps
    impala_times = np.array([r.get("_time", 0) for r in impala_rows], dtype=float)
    dreamer_times = np.array([r.get("_time", 0) for r in dreamer_rows], dtype=float)

    impala_hours = (impala_times - impala_times[0]) / 3600
    dreamer_hours = (dreamer_times - dreamer_times[0]) / 3600

    fig, ax = plt.subplots(figsize=(10, 5))
    if len(impala_hours) == len(impala_returns):
        ax.plot(impala_hours, smooth(impala_returns, 20),
                label="IMPALA", color="tab:blue", linewidth=2)
    if len(dreamer_hours) == len(dreamer_returns):
        ax.plot(dreamer_hours, smooth(dreamer_returns, 20),
                label="DreamerV2", color="tab:orange", linewidth=2)

    ax.set_xlabel("Wall-Clock Time (hours)")
    ax.set_ylabel("Episode Return")
    ax.set_title("Wall-Clock Efficiency: IMPALA vs DreamerV2")
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Need both IMPALA and DreamerV2 logs for wall-clock comparison")

## Next Steps

- **Full paper replication**: Train both algorithms for 100M steps with 128 actors/8 envs
- **Multi-size comparison**: Train on all 4 maze sizes (9x9 through 15x15)
- **Engine comparison**: See `05_engine_comparison.ipynb` for MuJoCo vs Genesis